In [ ]:
from src.config_manager import ConfigManager
from src.training_manager import TrainingManager
from src.experiment_manager import ExperimentManager
import torch
device = str(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(device)

## Choose dataset and models

First, we decide on a name for the experiment we will be conducting (all results and models from this experiment will be saved in a folder with the given name).

Then, we decide on which dataset we will use for the experiment. Options are "lorenz", "spiral", "double_pendulum", "random_skew","logistic_map", "hopf" and "pod".

Then, we decide on the models to train and compare for this experiment. The options are "mlp", "transformer", "rnnautoreg","oldrnn", "esn", "nodemlp","koopmanmlp", "nodetransformer", "koopmantransformer","nodernnautoreg","koopmanrnnautoreg", "none" (check readme for more info). We can choose to train each model a number of times, to compare the effect of different initializations.

In [1]:

dataset_name = "random_skew"
experiment_name = f"all_{dataset_name}_bests" 
models = [['none',1],['mlp',1]]



## Train the models

In [5]:
config_manager = ConfigManager(dataset_name, device)
training_manager = TrainingManager(device)

data_handler_params = config_manager.get_current_dataset_config()

In [ ]:
training_manager.train_multiple_models(experiment_name, config_manager, models)

### Create relative latent spaces and similarity matrix

To compute the relative latent spaces, we need to choose a number of anchors and a number of samples to embed in the relative latent space.

In [ ]:
anchors = 80
points_to_embed = 1000

experiment_manager = ExperimentManager(config_manager, device)

models, latent_spaces, absolute_latent_spaces = experiment_manager.evalute_latent_spaces_exp(experiment_name, points_to_embed, anchors, random_anchors=True)

### Benchmark the model performances
This will create a benchmark file in each models directory which will contain useful metrics that measure the models performance.

In [ ]:
experiment_manager.benchmark_models_exp(experiment_name)

### Visualize relative latent spaces
We will create plots that visualize the relative latent spaces of different modes.

For this, we will choose a method to reduce the dimensionality (either "pca" or "umap"), and a list of the directories of the models we want to include in the plot.

The plots will be saved in the experiment directory.

In [ ]:
#choose which models latent spaces to visualize
#specify the individual plots like this: vis_latent = (reducer, [model1, model2]), for reducers [pca, umap] are available
vis_latent1 = ('pca',['mlp1','transformer1','none1','rnnautoreg1','nodernnautoreg1','nodetransformer1','koopmantransformer1','koopmanrnnautoreg1','koopmanmlp1','esn1','oldrnn1','nodemlp1'])
vis_latent2 = ('umap', ['oldrnn1','mlp1', 'esn1', 'none1'])
vis_latent = [vis_latent1,vis_latent2]

experiment_manager.visualize_relative_latent_spaces(models, latent_spaces, vis_latent, experiment_name, fit_individually=False, n_components = 3)

### Visualize absolute latent spaces
It works the same as visualizing relative latent spaces but you need to pass the absolute_latent_spaces and
set fit_individually=True or fix_global_limits=True since the absolute latent spaces have different shapes.

In [ ]:
#choose which models latent spaces to visualize
#specify the individual plots like this: vis_latent = (reducetar, [model1, model2]), for reducers [pca, umap] are available

vis_latent1 = ('pca',['mlp1','transformer1','none1','rnnautoreg1','nodernnautoreg1','nodetransformer1','koopmantransformer1','koopmanrnnautoreg1','koopmanmlp1','esn1','oldrnn1','nodemlp1'])
vis_latent2 = ('umap', ['oldrnn1','mlp1', 'esn1', 'none1'])
vis_latent = [vis_latent1,vis_latent2]

                                                            #!                                                   #!                                       #!
experiment_manager.visualize_relative_latent_spaces(models, absolute_latent_spaces, vis_latent, experiment_name, fit_individually=True, n_components = 3, fix_global_limits = True)